## Webpage Extraction and Embedding (PolyU CUS)

### 1. Extracting raw text data

In [12]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [13]:
cus_URL = "https://www.polyu.edu.hk/cus/"
start_idx, stop_idx = 6450, -850

docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "lxml")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess whitespaces
    return text

cus_loader = RecursiveUrlLoader(
    url=cus_URL,
    base_url=cus_URL,
    prevent_outside=True,
    exclude_dirs=[
        cus_URL+"about-cus",
        cus_URL+"about-ous",
        cus_URL+"Sitemap", 
        cus_URL+"Search-Result", 
        cus_URL+"staff",
        cus_URL+"Staff",
        cus_URL+"internal",
        cus_URL+"nationaleducation",
        cus_URL+"Undergraduate-Studies-Support/Staff",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = cus_loader.lazy_load()
for doc in docs_lazy:
    #print(doc)
    doc.page_content = doc.page_content[start_idx:stop_idx]
    for key in unwanted_metadata:
        del doc.metadata[key]
    docs.append(doc)

In [14]:
print(f"Extracted number of webpages in CUS: {len(docs)}")
print(docs[15])

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in CUS: 35
page_content='            Impact study
                            

                                Contact Us
                            

                                Personal Information Collection Statement
                            

Quick Access

Start main content

													Home
												

													Student
												

													Senior Year Intakes and Articulation Degree Programme
												

													Academic Integrity
												

Academic Integrity

Academic integrity (學術誠信) refers to the honest and ethical manner in which academic work is done, whether it is an assignment, an examination, an oral presentation, or a research project or report.
PolyU views Plagiarism as a serious disciplinary offence.
It is a fundamental value that all students at PolyU are expected to uphold.
As part of the University’s continuous effort in maintaining a fair and honest learning environment at PolyU, an online tutorial has 

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_CUS_{url_end}_chunk_{i}"

In [16]:
print(chunks[20])

page_content='The test contains four parts:

Listening Tasks (30 minutes)
Reading Tasks (60 minutes)
Writing Tasks (60 minutes)
Speaking Tasks (11-14 minutes)

Sponsorship Scheme to graduating students of undergraduate degree programmes to take the International English Language Testing System (IELTS) 2025/26
If you are a final year undergraduate degree student, you are invited to participate in the IELTS test sessions under the sponsorship scheme. The IELTS test can let you know more about your performance on your English language after these few years of study in PolyU. 
Information and arrangements for the test sessions in Semester one 2025/26 is provided below.

Participation
Students’ participation in this Sponsorship Scheme is voluntary, and the results obtained will not affect their graduation.' metadata={'source': 'https://www.polyu.edu.hk/cus/ielts/', 'content_type': 'text/html; charset=utf-8', 'title': 'IELTS | College of Undergraduate Studies', 'description': '“Effective com

### 3. Document Embedding in Chroma

In [17]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_cus_webpage" if not SINGLE else "vaa_documents"

In [18]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

In [19]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i + 0)]
    )

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 123 chunks into ChromaDB to vaa_documents


### 4. Simple Testing

In [20]:
query = "What is the general university requirement for undergraduate student?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: Impact study
                            

                                Contact Us
                            

                                Personal Information Collection Statement
                            

Quick Access

Start main content

													Home
												

													Student
												

													4-Year Undergraduate Student
												

													General University Requirements (GUR)
												

General University Requirements (GUR)

GUR for 4-Year Undergraduate Student 

 

Admitted in 2021/22 or before

Freshman Seminar

Language & Communication Requirements

Leadership & Intra-Personal Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

Admitted from 2022/23

Artificial Intelligence and Data Analytics Requirement

Innovation and Entrepreneurship Requirement

Language & Communication Requirements

Leadership Education and Development

Cluster-Area Requirements

Service-Learning...
Source: https://www.poly